# VoxEntity-EU — generation notebook (thin; logic lives in the package)

Installs the pinned environment, loads templates, builds the corpus, verifies rights, generates synthesis batches with per-clip checkpointing, and runs automated QA. All heavy lifting is in `voxentity_eu.*`; this notebook only orchestrates.

In [ ]:
import subprocess
import sys

# Core install (adjust index for CUDA torch as documented in README)
print(subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.[tts,dev]'], cwd='../..').returncode)

In [ ]:
# 1. corpus build + validation
import sys

from voxentity_eu.cli import main as cli_main

rc = cli_main(['corpus', 'build', '--config', 'configs/corpus.yaml', '--out', 'data/generated'])
rc = cli_main(['corpus', 'validate', '--manifest', 'data/generated/audio_manifest.jsonl', '--gold', 'data/generated/gold/scenarios.jsonl'])
assert rc == 0, 'corpus validation failed'

In [ ]:
# 2. rights registry — every asset must be resolved before audio redistribution
from pathlib import Path

from voxentity_eu.corpus.rights import write_registry

entries = write_registry(Path('data/generated/rights_registry.jsonl'))
for e in entries:
    print(e.source_id, e.review_status, e.redistribution)

In [ ]:
# 3. synthesis (GPU, ~9 s/clip on a GTX 1080 with fp32; resume-safe)
# Run via CLI so stdout stays small; logs print every 20 clips.
print(subprocess.run([sys.executable, '-m', 'voxentity_eu.cli', 'synthesize',
    '--manifest', 'data/generated/audio_manifest.jsonl', '--profile', 'qwen', '--resume']).returncode)

In [ ]:
# 4. automated QA + telephone channel
print(cli_main(['qa', '--manifest', 'data/generated/audio_manifest.jsonl', '--out', 'data/generated/qa.jsonl']))
print(cli_main(['transform', '--manifest', 'data/generated/audio_manifest.jsonl', '--recipe', 'configs/channel.yaml', '--resume']))

In [ ]:
# 5. gold-free inputs export (opaque ids; mapping stays gold-side)
print(cli_main(['corpus', 'export-inputs', '--manifest', 'data/generated/audio_manifest.jsonl', '--out', 'data/inputs']))